# Parcial para la casa · Taylor — por qué el GPS de tu celular necesita a Einstein

**Métodos Numéricos · UNAL Manizales · 2026-2**

Los relojes atómicos de los satélites GPS no marchan al mismo ritmo que los relojes en tierra: la velocidad los **atrasa** (relatividad especial) y estar más arriba en el campo gravitatorio los **adelanta** (relatividad general). Los dos efectos son del orden de 10⁻¹⁰, y para calcularlos bien en un computador hace falta… el **polinomio de Taylor**. En este cuaderno:

1. lees elementos orbitales reales de satélites GPS (y de la Estación Espacial Internacional),
2. calculas cuántos microsegundos por día gana el reloj del satélite y lo comparas con el número oficial del sistema GPS,
3. ves en vivo la **cancelación catastrófica** de la fórmula directa y cómo Taylor la arregla,
4. traduces microsegundos a kilómetros de error en tu posición,
5. auditas al agente de la Parte III.

**Física como receta (no tienes que saber relatividad).** Con x = v²/c²:

- ritmo perdido por velocidad: γ − 1, con γ = 1/√(1 − x);
- ritmo ganado por gravedad: GM·(1/R_T − 1/r)/c²;
- radio de la órbita a partir de las vueltas por día n (tercera ley de Kepler): r = (GM/ω²)^(1/3), con ω = 2πn/86400 rad/s, y velocidad v = √(GM/r).

**Fuentes.**
- *Elementos orbitales (TLE)* del catálogo público de la Fuerza Espacial de EE. UU. (18th/19th Space Defense Squadron), redistribuidos por **CelesTrak** (T. S. Kelso), `https://celestrak.org/NORAD/elements/gp.php`. La celda 1 los consulta en vivo. **Copia congelada:** las líneas TLE de los mismos satélites, copiadas textualmente de N2YO.com (que muestra los TLE de la misma fuente, "AFSPC") el **15 de octubre de 2026** y verificadas con el dígito de control de cada línea. **Uso:** datos públicos de libre consulta; cita la fuente ("CelesTrak / Fuerza Espacial de EE. UU., consultado el 15/10/2026").
- *Constantes:* GM = 3,986004418·10¹⁴ m³/s², R_T = 6 378 137 m (Ashby, N., 2003, "Relativity in the Global Positioning System", *Living Reviews in Relativity* 6:1); c = 299 792 458 m/s (exacta).
- *Valores oficiales:* IS-GPS-200 (especificación de la interfaz GPS, gps.gov): frecuencia nominal 10,23 MHz, desplazada en Δf/f = −4,4647·10⁻¹⁰, es decir 10,2299999954326 MHz; constante F = −4,442807633·10⁻¹⁰ s/√m con μ = 3,986005·10¹⁴ m³/s². ESA Navipedia ("Relativistic Clock Correction"): el reloj del satélite se ve adelantarse ≃ 38 μs/día.

Reglas del cuaderno: solo cambias lo marcado `# <-- CAMBIA ESTO`. Antes de cada experimento hay una caja **Predice en tu guía antes de correr**: escribe tu predicción en papel *antes* de ejecutar. Ojo: Python imprime con punto decimal.

## Celda 1 · Carga de datos reales (CelesTrak en vivo + respaldo congelado)

Usamos tres satélites GPS (lanzados en 1997, 2018 y 2021) y la Estación Espacial Internacional (EEI). De cada uno necesitamos dos números: la **excentricidad** e y el **movimiento medio** n (vueltas por día).

Si `USAR_EN_VIVO = True`, la celda intenta CelesTrak; si falla (sin internet, sitio caído), usa la copia congelada. Si usas datos en vivo, tus cifras pueden diferir de las de la guía en las últimas cifras (los satélites se mueven y los TLE se actualizan): anota las tuyas.

In [ ]:
import io, math, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import mpmath as mp                       # viene instalado en Colab
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mpmath"])
    import mpmath as mp
mp.mp.dps = 50                                # 50 cifras: nuestra "verdad"

USAR_EN_VIVO = True                           # <-- CAMBIA ESTO (False = solo copia congelada)

# Número de catálogo NORAD de cada objeto
OBJETOS = {"NAVSTAR 43": 24876, "NAVSTAR 77": 43873, "NAVSTAR 81": 48859, "EEI": 25544}

# Copia congelada (15/10/2026): línea 2 de cada TLE, copiada textualmente.
# (los espacios internos se colapsaron al copiar; por eso se lee con split())
RESPALDO_TLE = {
    "NAVSTAR 43": "2 24876 56.0660 93.9155 0106459 59.3258 301.7338 2.00564660214349",
    "NAVSTAR 77": "2 43873 55.7464 88.3292 0040402 196.2470 336.0562 2.00565579 57038",
    "NAVSTAR 81": "2 48859 55.1085 327.6294 0029210 232.3643 126.6868 2.00562236 39151",
    "EEI":        "2 25544 51.6325 73.0089 0006711 259.3258 100.6975 15.48855783590141",
}
RESPALDO_EPOCA = {"NAVSTAR 43": "26287.36219116", "NAVSTAR 77": "26265.97611495",
                  "NAVSTAR 81": "26288.16502826", "EEI": "26286.82178372"}  # año 2026, día del año

def leer_linea2(linea):
    """Excentricidad (campo 5, punto decimal implícito) y movimiento medio (campo 8, 8 decimales)."""
    t = linea.split()
    e = float("0." + t[4])
    entero, dec = t[7].split(".")
    n = float(entero + "." + dec[:8])     # lo que sigue son el número de vuelta y el dígito de control
    return e, n

def consultar_celestrak(catnr):
    url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={catnr}&FORMAT=csv"
    req = urllib.request.Request(url, headers={"User-Agent": "parcial-metodos-unal"})
    with urllib.request.urlopen(req, timeout=20) as r:
        tabla = pd.read_csv(io.StringIO(r.read().decode("utf-8")))
    fila = tabla.iloc[0]
    return float(fila["ECCENTRICITY"]), float(fila["MEAN_MOTION"]), str(fila["EPOCH"])

filas = []
for nombre, catnr in OBJETOS.items():
    e_c, n_c = leer_linea2(RESPALDO_TLE[nombre])
    e, n, epoca, origen = e_c, n_c, "2026, día " + RESPALDO_EPOCA[nombre], "respaldo congelado"
    if USAR_EN_VIVO:
        try:
            e_v, n_v, epoca_v = consultar_celestrak(catnr)
            if abs(n_v - n_c)/n_c < 0.01 and 0 <= e_v < 0.05:   # control de cordura
                e, n, epoca, origen = e_v, n_v, epoca_v, "CelesTrak en vivo"
        except Exception:
            pass
    filas.append(dict(objeto=nombre, e=e, n_vueltas_dia=n, epoca=epoca, origen=origen,
                      e_congelado=e_c, n_congelado=n_c))
datos = pd.DataFrame(filas).set_index("objeto")
pd.set_option("display.precision", 10)
datos

## Celda 2 · Ítem II.1 — ¿Qué tan rápido va un satélite GPS?

Receta: ω = 2πn/86400, r = (GM/ω²)^(1/3), v = √(GM/r), x = v²/c².

**Predice en tu guía antes de correr:** un satélite GPS da 2 vueltas por día a r ≈ 26 560 km del centro de la Tierra. Calcula a mano v ≈ 2πr / (medio día) en km/s y luego x = v²/c². ¿La EEI (16 vueltas al día, mucho más baja) va más rápido o más despacio? ¿Su x será mayor o menor que el del GPS?

In [ ]:
GM  = 3.986004418e14     # m^3/s^2  (Ashby 2003)
c   = 299792458.0        # m/s (exacta)
R_T = 6378137.0          # m, radio ecuatorial de la Tierra (WGS 84)
DIA = 86400.0            # s

def orbita(n_vueltas_dia):
    """Radio r (m), velocidad v (m/s) y x = v^2/c^2 a partir de las vueltas por día."""
    w = 2*math.pi*n_vueltas_dia/DIA
    r = (GM/w**2)**(1/3)
    v = math.sqrt(GM/r)
    return r, v, v**2/c**2

for nombre in datos.index:
    r, v, x = orbita(datos.loc[nombre, "n_vueltas_dia"])
    datos.loc[nombre, "r_km"] = r/1000
    datos.loc[nombre, "altura_km"] = (r - R_T)/1000
    datos.loc[nombre, "v_km_s"] = v/1000
    datos.loc[nombre, "x = v^2/c^2"] = x
with pd.option_context("display.float_format", "{:.6g}".format, "display.width", 200):
    print(datos[["n_vueltas_dia", "r_km", "altura_km", "v_km_s", "x = v^2/c^2"]])

## Celda 3 · Ítem II.2 — Einstein en microsegundos por día

Ritmo relativo del reloj del satélite respecto a uno en tierra (aproximación de primer orden):

- gana por gravedad: GM·(1/R_T − 1/r)/c²
- pierde por velocidad: γ − 1 ≈ x/2 (Taylor de grado 1, que justificaste en la Parte I)

Multiplicando por 86 400 s y por 10⁶ se obtienen microsegundos por día.

**Predice en tu guía antes de correr:** (a) ¿cuál efecto gana en el GPS, gravedad o velocidad? (b) ¿Cuántos μs/día netos esperas, comparado con los ≈ 38 μs/día que cita la ESA? (c) ¿Y en la EEI, que está a solo ~420 km de altura y va al doble de velocidad: se adelanta o se atrasa su reloj?

In [ ]:
def microsegundos_por_dia(r, x):
    grav = GM*(1/R_T - 1/r)/c**2     # adelanto por gravedad
    vel  = x/2                        # atraso por velocidad (Taylor de grado 1)
    return grav*DIA*1e6, vel*DIA*1e6, (grav - vel)*DIA*1e6, grav - vel

filas = []
for nombre in datos.index:
    r, v, x = orbita(datos.loc[nombre, "n_vueltas_dia"])
    g, ve, neto, frac = microsegundos_por_dia(r, x)
    filas.append(dict(objeto=nombre, gravedad_us_dia=g, velocidad_us_dia=-ve, neto_us_dia=neto, df_f=frac))
efectos = pd.DataFrame(filas).set_index("objeto")
with pd.option_context("display.float_format", "{:.6g}".format, "display.width", 200):
    print(efectos)

# Lo oficial (IS-GPS-200): el reloj se fabrica con df/f = -4.4647e-10 para compensar
DF_F_OFICIAL = -4.4647e-10
print("\nIS-GPS-200: df/f =", DF_F_OFICIAL, "->", -DF_F_OFICIAL*DIA*1e6, "us/día de compensación")
print("Frecuencia oficial del reloj del satélite: 10.23 MHz * (1 + df/f) =", f"{10.23e6*(1 + DF_F_OFICIAL):.7f} Hz")
frac81 = efectos.loc["NAVSTAR 81", "df_f"]
print("Con NUESTRO cálculo (NAVSTAR 81): df/f =", f"{-frac81:.5e}", "-> frecuencia", f"{10.23e6*(1 - frac81):.7f} Hz")
print("Diferencia relativa con lo oficial:", f"{abs(frac81 + DF_F_OFICIAL)/abs(DF_F_OFICIAL)*100:.2f} %")

## Celda 4 · Tu función: γ − 1 con el polinomio de Taylor

Los coeficientes de Taylor de 1/√(1 − x) alrededor de 0 son 1, 1/2, 3/8, 5/16, 35/128, …; cada uno sale del anterior multiplicando por (2k − 1)/(2k). Entonces

γ − 1 ≈ (1/2)x + (3/8)x² + (5/16)x³ + …  (grado `grado`).

**Para completar (opcional, como en los labs):** escribe `gamma_menos_1_taylor(x, grado)`. Si la dejas devolviendo `None`, la celda 4b usa la versión de respaldo.

In [ ]:
def gamma_menos_1_taylor(x, grado):
    # COMPLETA: suma de k = 1 hasta grado de coef_k * x**k, con coef_1 = 1/2 y
    # coef_k = coef_{k-1} * (2k-1)/(2k)
    return None  # <-- CAMBIA ESTO

### Celda 4b · Respaldo (no la modifiques)

In [ ]:
def _taylor_respaldo(x, grado):
    coef, suma, potencia = 1.0, 0.0, 1.0
    for k in range(1, grado + 1):
        coef *= (2*k - 1)/(2*k)
        potencia *= x
        suma += coef*potencia
    return suma

if gamma_menos_1_taylor(0.01, 3) is None:
    gamma_menos_1_taylor = _taylor_respaldo
    print("Usando la función de Taylor de respaldo.")
else:
    print("Tu función con x=0.01, grado 3:", gamma_menos_1_taylor(0.01, 3),
          "| respaldo:", _taylor_respaldo(0.01, 3))

## Celda 5 · Ítem II.3 — Cancelación catastrófica en vivo

Tres maneras de calcular γ − 1 para el satélite GPS, comparadas con la "verdad" en 50 cifras (mpmath):

- **directa:** `1/math.sqrt(1 - x) - 1` (doble precisión, 64 bits)
- **Taylor grado 1 y grado 2** (tu función)
- **directa en float32** (precisión simple, 32 bits, la de muchas GPU)

"Cifras correctas" = −log₁₀(error relativo).

**Predice en tu guía antes de correr:** el épsilon de máquina en doble precisión es ≈ 2,2·10⁻¹⁶ y x ≈ 1,7·10⁻¹⁰. Cuando calculas 1 − x y luego restas 1, ¿cuántas de las ~16 cifras de la doble precisión crees que sobreviven? ¿Qué pasa en float32, cuyo épsilon es ≈ 1,2·10⁻⁷?

In [ ]:
objeto = "NAVSTAR 81"                           # <-- CAMBIA ESTO ("NAVSTAR 43", "NAVSTAR 77", "EEI")
r, v, x = orbita(datos.loc[objeto, "n_vueltas_dia"])

verdad = 1/mp.sqrt(1 - mp.mpf(x)) - 1           # 50 cifras
def cifras(aprox):
    err = abs((mp.mpf(float(aprox)) - verdad)/verdad)
    return float(err), (float(-mp.log10(err)) if err > 0 else float("inf"))

x32 = np.float32(x)
candidatos = {
    "directa (float64)":    1/math.sqrt(1 - x) - 1,
    "Taylor grado 1":       gamma_menos_1_taylor(x, 1),
    "Taylor grado 2":       gamma_menos_1_taylor(x, 2),
    "expm1(-log1p(-x)/2)":  np.expm1(-0.5*np.log1p(-x)),      # funciones de biblioteca "anti-cancelación"
    "directa (float32)":    float(np.float32(1)/np.sqrt(np.float32(1) - x32) - np.float32(1)),
}
print(f"{objeto}: x = {x:.10e}")
print(f"verdad (mpmath):        gamma-1 = {mp.nstr(verdad, 20)}")
for nombre, val in candidatos.items():
    err, cif = cifras(val)
    print(f"{nombre:22s} gamma-1 = {val:.16e}   error relativo = {err:.2e}   cifras correctas = {cif:.1f}")
print("\n¿En float32 es 1 - x igual a 1?", np.float32(1) - x32 == np.float32(1))
print("eps float64 =", np.finfo(np.float64).eps, "  eps float32 =", np.finfo(np.float32).eps)
print("\nEn microsegundos por día, el atraso por velocidad sería:")
for nombre, val in candidatos.items():
    print(f"  {nombre:22s} {val*DIA*1e6:.10f} us/día")
print(f"  {'verdad':22s} {float(verdad)*DIA*1e6:.10f} us/día")

## Celda 6 · Ítem II.3 (continuación) — La "V" del error

Recorremos x desde 10⁻¹⁴ hasta 10⁻¹ y graficamos el error relativo de cada fórmula contra la verdad (mpmath).

**Predice en tu guía antes de correr:** dibuja a mano, en escala log-log, cómo crees que se ve el error de la fórmula directa (¿sube o baja con x?) y el de Taylor de grado 1 (¿sube o baja con x? ¿con qué pendiente, si el resto es ≈ (3/8)x²?). ¿Dónde se cruzan?

In [ ]:
xs = np.logspace(-14, -1, 300)
def err_rel(f):
    out = []
    for xx in xs:
        V = 1/mp.sqrt(1 - mp.mpf(xx)) - 1
        out.append(float(abs((mp.mpf(f(xx)) - V)/V)))
    return np.maximum(np.array(out), 1e-18)   # para que el 0 se vea en escala log

e_dir = err_rel(lambda t: 1/math.sqrt(1 - t) - 1)
e_t1  = err_rel(lambda t: gamma_menos_1_taylor(t, 1))
e_t2  = err_rel(lambda t: gamma_menos_1_taylor(t, 2))

plt.figure(figsize=(8, 5))
plt.loglog(xs, e_dir, label="directa 1/sqrt(1-x) - 1", lw=1)
plt.loglog(xs, e_t1, label="Taylor grado 1", lw=2)
plt.loglog(xs, e_t2, label="Taylor grado 2", lw=2)
plt.axhline(1e-12, color="gray", ls=":", label="tolerancia 1e-12 (Parte III)")
for nombre in datos.index:
    plt.axvline(orbita(datos.loc[nombre, "n_vueltas_dia"])[2], color="k", alpha=0.2)
plt.xlabel("x = v²/c²"); plt.ylabel("error relativo de gamma - 1")
plt.title("Cancelación (directa) vs truncamiento (Taylor)"); plt.legend(loc="lower right", fontsize=8); plt.grid(alpha=0.3)
plt.show()

cruce = xs[np.argmax(e_t1 > e_dir)]
print(f"Taylor grado 1 empieza a perder contra la directa hacia x ≈ {cruce:.1e} (aprox.: la curva azul es ruidosa)")
print(f"Taylor grado 1 deja de cumplir 1e-12 hacia x ≈ {xs[np.argmax(e_t1 > 1e-12)]:.2e}  (v ≈ {c*math.sqrt(xs[np.argmax(e_t1 > 1e-12)]):.0f} m/s)")

## Celda 7 · Ítem II.4 — De microsegundos a kilómetros

El receptor de tu celular calcula su distancia a cada satélite como c·Δt. Un error de reloj Δt produce un error de distancia c·Δt.

Además, como la órbita no es un círculo perfecto (e > 0), el satélite sube y baja un poco y el efecto relativista oscila. La IS-GPS-200 manda que el receptor corrija Δt_r = F·e·√a·sen E, con F = −2√μ/c² (μ = 3,986005·10¹⁴ en la especificación). La amplitud es |F|·e·√a.

**Predice en tu guía antes de correr:** (a) si nadie corrigiera los ≈ 38 μs/día, ¿cuántos km de error acumularías en un día? ¿Y en un minuto? (b) ¿Cuál de los tres satélites tendrá la corrección periódica más grande, mirando solo su e?

In [ ]:
horas = 24                                     # <-- CAMBIA ESTO (prueba 1/60, 1, 24)
nombre = "NAVSTAR 81"
r, v, x = orbita(datos.loc[nombre, "n_vueltas_dia"])
frac = GM*(1/R_T - 1/r)/c**2 - x/2
dt = frac*horas*3600
print(f"{nombre}: en {horas:.4g} h el reloj se adelanta {dt*1e6:.3f} us -> error de distancia c*dt = {c*dt/1000:.3f} km = {c*dt:.1f} m")

# Constante F de la IS-GPS-200, recalculada
mu_gps = 3.986005e14
F = -2*math.sqrt(mu_gps)/c**2
print(f"\nF = -2*sqrt(mu)/c^2 = {F:.9e}   (IS-GPS-200: -4.442807633e-10)")

print("\nCorrección periódica por excentricidad (amplitud |F|*e*sqrt(a)):")
for nombre in ["NAVSTAR 43", "NAVSTAR 77", "NAVSTAR 81"]:
    e = datos.loc[nombre, "e"]
    a = orbita(datos.loc[nombre, "n_vueltas_dia"])[0]
    amp = abs(F)*e*math.sqrt(a)
    print(f"  {nombre}: e = {e:.7f}  amplitud = {amp*1e9:.2f} ns  ->  {amp*c:.2f} m")
print(f"  (Ashby 2003: para e = 0.01 la amplitud es 'about 23 ns'; con a = 26561.75 km da {abs(F)*0.01*math.sqrt(2.656175e7)*1e9:.2f} ns)")

## Celda 8 · Parte III — Auditor del agente

> **Escenario inventado para el parcial.** El agente no existe; esta celda calcula de verdad la tabla que se le atribuye.

Para cada objeto, cada acción y su costo en operaciones de punto flotante:

- `directa`: 1/√(1−x) − 1 → 4 operaciones (resta, raíz, división, resta)
- `reescrita`: x / (√(1−x)·(1+√(1−x))) → 5 operaciones
- `Taylor n` con Horner: x·(c₁ + x·(c₂ + … + x·cₙ)) → 2n − 1 operaciones

Recompensa = −(operaciones) − `castigo` si el error relativo supera `tol`. El error se mide contra `referencia`.

**Predice en tu guía antes de correr:** ¿qué acción crees que elige el agente para el avión, el GPS y el muón? Para el ítem III.2 b) pon `v_extra = 600` (y prueba también 300); para III.3 cambia `referencia` y `castigo`.

In [ ]:
referencia = "mpmath"        # <-- CAMBIA ESTO ("mpmath" o "directa")
tol = 1e-12                  # <-- CAMBIA ESTO
castigo = 1000               # <-- CAMBIA ESTO
v_extra = None               # <-- CAMBIA ESTO (una velocidad en m/s para agregar un objeto, p. ej. 600)

def x_desde_gamma(g): return 1 - 1/g**2
casos = {
    "Avión Boeing 787 (903 km/h)":    (903/3.6)**2/c**2,
    "Satélite GPS NAVSTAR 81":        orbita(datos.loc["NAVSTAR 81", "n_vueltas_dia"])[2],
    "Estación Espacial Internacional": orbita(datos.loc["EEI", "n_vueltas_dia"])[2],
    "Sonda Parker (191 km/s)":        (191e3)**2/c**2,
    "Electrón de TV de tubo (25 kV)": x_desde_gamma(1 + 25/510.99895),       # m_e c^2 = 0.51099895 MeV
    "Muón de rayo cósmico (4 GeV)":   x_desde_gamma(4000/105.6583755),       # m_mu c^2 = 105.6583755 MeV
}
if v_extra is not None:
    casos[f"Objeto extra (v = {v_extra} m/s)"] = (v_extra/c)**2
def taylor_horner(t, n):
    """x*(c1 + x*(c2 + ... + x*cn)): n multiplicaciones y n-1 sumas = 2n-1 operaciones."""
    cs = [1.0]
    for k in range(1, n + 1): cs.append(cs[-1]*(2*k - 1)/(2*k))
    p = cs[n]
    for k in range(n - 1, 0, -1): p = cs[k] + t*p
    return t*p
def directa(t):   return 1/math.sqrt(1 - t) - 1
def reescrita(t): s = math.sqrt(1 - t); return t/(s*(1 + s))
acciones = [("directa", 4, directa), ("reescrita", 5, reescrita)] + \
           [(f"Taylor {n}", 2*n - 1, (lambda n: lambda t: taylor_horner(t, n))(n)) for n in range(1, 21)]

filas = []
for nombre, x in casos.items():
    V = 1/mp.sqrt(1 - mp.mpf(x)) - 1
    ref = V if referencia == "mpmath" else mp.mpf(directa(x))
    mejor, info = None, {}
    for accion, ops, f in acciones:
        err = float(abs((mp.mpf(f(x)) - ref)/ref))
        rec = -ops - (castigo if err > tol else 0)
        info[accion] = err
        if mejor is None or rec > mejor[2]:
            mejor = (accion, ops, rec)
    grado_min = next((n for n in range(1, 21) if info[f"Taylor {n}"] <= tol), "> 20")
    filas.append({"objeto": nombre, "x": x,
                  "err directa": info["directa"], "err Taylor 1": info["Taylor 1"],
                  "err Taylor 2": info["Taylor 2"], "err Taylor 20": info["Taylor 20"],
                  "err reescrita": info["reescrita"], "grado mín. Taylor": grado_min,
                  "acción aprendida": mejor[0], "recompensa": mejor[2]})
tabla = pd.DataFrame(filas).set_index("objeto")
with pd.option_context("display.float_format", "{:.2e}".format, "display.width", 250, "display.max_columns", 20):
    print(f"referencia = {referencia}, tol = {tol}, castigo = {castigo}\n")
    print(tabla)

# ¿Cuántos términos necesitaría Taylor alrededor de 0 para el muón? (suma exacta con mpmath)
mp.mp.dps = 30
xm = mp.mpf(casos["Muón de rayo cósmico (4 GeV)"]); Vm = 1/mp.sqrt(1 - xm) - 1
coef, suma, pot, k = mp.mpf(1), mp.mpf(0), mp.mpf(1), 0
while abs((Vm - suma)/Vm) > tol:
    k += 1; coef *= mp.mpf(2*k - 1)/(2*k); pot *= xm; suma += coef*pot
print(f"\nMuón: Taylor alrededor de 0 necesita grado {k} para error relativo <= {tol}")
mp.mp.dps = 50